In [2]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols
from scipy.stats import shapiro, levene

In [3]:
# Étape 1: Charger les données
# Supposons que vos données sont dans un fichier CSV nommé 'data.csv'
data = pd.read_csv('/Users/rudy/Documents/CHU/iias/baiddy_group/automl/src/perf_logger/tests_data/bench_v2.log')
data['perf'] = data.apply(lambda row: row['balanced_accuracy'] if pd.notnull(row['balanced_accuracy']) else row['r2_score'], axis=1)
data = data[(data['max_duration'] - data['compute_time']) >= -30] # remove line with too long computing time
data = data[~data['package'].isin(['automed_old', 'automed_old2', 'IAML1.0'])]

In [4]:


# Étape 2: Préparer les données
# Assurons-nous que les colonnes sont correctement formatées comme catégorielles si nécessaire
data['package'] = data['package'].astype('category')
data['dataset'] = data['dataset'].astype('category')
data['max_duration'] = data['max_duration'].astype('category')

# Étape 3: Effectuer l'ANOVA à trois facteurs
# Formule: 'performance ~ C(model) + C(dataset) + C(max_duration) + C(model):C(dataset) + C(model):C(max_duration) + C(dataset):C(max_duration) + C(model):C(dataset):C(max_duration)'
model = ols('performance ~ C(package) * C(dataset) * C(max_duration)', data=data).fit()
anova_results = sm.stats.anova_lm(model, typ=2)  # Utiliser typ=2 pour l'ANOVA à effets mixtes
print(anova_results)

# Étape 4: Tester les hypothèses de l'ANOVA
# Normalité des résidus
_, p_normality = shapiro(model.resid)
print("Test de Shapiro-Wilk pour la normalité des résidus (p-value):", p_normality)

# Homogénéité des variances
# Utiliser Levene ou Bartlett selon la normalité des données
grouped_data = data.groupby(['package', 'dataset', 'max_duration'])
_, p_homogeneity = levene(*[group['performance'] for name, group in grouped_data])
print("Test de Levene pour l'homogénéité des variances (p-value):", p_homogeneity)

# S'assurer que p_normality et p_homogeneity sont supérieurs à 0.05 pour valider les hypothèses
if p_normality > 0.05 and p_homogeneity > 0.05:
    print("Les hypothèses de l'ANOVA sont validées.")
else:
    print("Les hypothèses de l'ANOVA ne sont pas validées, considérez des transformations ou des tests non paramétriques.")


KeyError: 'model'